In [274]:
TGT_CWD = '~/Documents/repos/binance-public-data/python'
import pathlib; import sys

import ibis.interactive; sys.path.append(pathlib.Path(TGT_CWD).expanduser().as_posix())

from rsch.notebook_setup import *
from rsch.lakeshack import LakeShack
import rsch.etl as etl
import rsch.signals as signals
from rsch.universe import Universe, perf_stats

import pyarrow as pa

import ibis
from ibis import _
import ibis.selectors as s
ibis.interactive = True

In [275]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
pio.templates.default = 'plotly'
pd.options.plotting.backend = 'plotly'

In [276]:
# ls_path = TGT_CWD + '/data/lakeshack_v2'

ls_path = '/Users/tyler/Documents/repos/binance-public-data/python/data/lakeshack_v3'
ls_path = '/Users/tyler/Documents/repos/binance-public-data/python/data/lakeshack'
ls = LakeShack(ls_path)
ls._update_tables()

unv = Universe(
    bar=ls.to_ibis('um_klines_monthly_1d'),
    selection_secid_count=20
    # bar=ls.to_ibis('um.klines.monthly.1d'),
    
)


apply_universe = True
start_date = '2020-04-01'
# end_date = '2020-12-31'
end_date = '2020-04-30'
# end_date = '2020-06-30'

bar = (
    ls.
    # to_ibis('um.klines.monthly.1m')
    to_ibis('um_klines_monthly_1m')
    .filter(
        (_.close_time <= pd.Timestamp(end_date)) & \
        (_.close_time >= pd.Timestamp(start_date))
    )
)

unvm = Universe(
    bar=bar,
    weights=unv.weights,
    risk_calculation_window=180
)

# unvm = Universe(
#     bar=ls.to_ibis('um.klines.monthly.1m').filter((_.close_time <= pd.Timestamp(end_date))), # & (_.close_time >= pd.Timestamp(start_date)))),
#     weights=unv.weights
# )

/Users/tyler/Documents/repos/binance-public-data/python/rsch/universe.py:192: FutureWarning:

`Table.fillna` is deprecated as of v9.1; use fill_null instead



In [277]:
rt = (
    unvm.bar
    .filter(_['universe_mask'])
    .select(['open_time', 'secid', 'return'])
)

rt = rt.execute().pivot_table(index='open_time', columns='secid', values='return')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [278]:
rt = rt.dropna(axis=1)

In [279]:
periods = pd.period_range(rt.index.min(), rt.index.max(), freq='3D')

p = periods[0]

rt_p = rt.loc[p.start_time:p.end_time].dropna(axis=1, thresh=.8)
corr_p = rt_p.cov()
corr_p = corr_p.fillna(0)
eigvals, eigvecs = np.linalg.eigh(corr_p)

# reverse the order of the eigveactors and values
eigvals = eigvals[::-1]
eigvecs = eigvecs[:, ::-1]

# now make series and dataframe
eigvals = pd.Series(eigvals)
eigvecs = pd.DataFrame(eigvecs, index=corr_p.columns)

ev_rt = rt_p @ eigvecs
ev_rt_rm_60 = ev_rt.rolling(60).mean()

In [280]:
ev_rt.var()

0    0.0000
1    0.0000
2    0.0000
3    0.0000
4    0.0000
5    0.0000
6    0.0000
7    0.0000
8    0.0000
9    0.0000
10   0.0000
11   0.0000
12   0.0000
13   0.0000
14   0.0000
15   0.0000
16   0.0000
17   0.0000
dtype: float64

In [281]:
eigvals.mul(1e6)
eigvecs

0    19.5779
1     1.3114
2     0.8966
3     0.8770
4     0.8271
5     0.8040
6     0.7346
7     0.6233
8     0.6047
9     0.5599
10    0.5111
11    0.4610
12    0.4425
13    0.3556
14    0.2972
15    0.2793
16    0.2637
17    0.2518
dtype: float64

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
secid,,,,,,,,,,,,,,,,,,
ADAUSDT,0.2041,-0.0166,-0.1256,0.0079,0.0073,-0.0296,0.0932,-0.0214,-0.1805,0.2115,0.0015,0.8410,-0.1515,-0.0822,-0.2949,0.0655,0.1584,-0.0391
ATOMUSDT,0.2178,-0.0587,-0.7558,-0.1284,-0.1575,-0.4621,-0.1442,0.1776,0.1677,-0.1112,0.0780,-0.1208,0.0224,-0.0282,-0.0553,0.0529,-0.0002,-0.0490
BCHUSDT,0.2627,-0.1192,0.2780,-0.0420,-0.2145,0.0334,-0.1446,0.2558,0.1645,-0.1715,-0.0394,-0.0497,0.0076,-0.6380,-0.2456,-0.4000,0.0937,0.0867
BNBUSDT,0.2256,-0.0406,-0.0033,-0.1017,-0.0461,0.0664,-0.0677,0.0994,0.1054,0.8294,-0.3007,-0.2528,-0.0959,0.1208,-0.0648,-0.1419,0.0126,-0.1201
BTCUSDT,0.2989,-0.0658,0.0797,-0.1184,-0.0477,0.0779,-0.0472,0.0156,0.0506,0.0569,-0.1092,-0.0444,0.1070,-0.0948,0.1695,0.5944,0.2220,0.6354
DASHUSDT,0.2453,-0.1751,0.0775,0.6483,0.4204,-0.2823,-0.4293,-0.1325,-0.1194,0.0082,-0.0832,-0.0408,0.0250,-0.0066,0.0204,-0.0180,-0.0025,0.0137
EOSUSDT,0.2512,-0.1091,0.1421,-0.0384,-0.2216,0.0607,-0.1408,0.1216,0.0447,-0.2114,-0.0682,0.1165,0.3276,0.6847,-0.2727,-0.2409,-0.0763,0.1915
ETCUSDT,0.1865,-0.0905,0.1634,0.0077,-0.1238,-0.0345,-0.0167,0.1117,-0.0121,-0.1814,0.0981,-0.1127,-0.8764,0.2209,-0.0420,0.1157,-0.1143,0.0244
ETHUSDT,0.2730,-0.0893,0.1473,-0.0121,-0.1222,0.0882,-0.0463,0.1205,0.0469,-0.0077,0.0074,0.1156,0.2033,-0.1333,0.0987,0.3955,-0.6797,-0.3915


In [282]:
corr_p.mul(1e6)

secid,ADAUSDT,ATOMUSDT,BCHUSDT,BNBUSDT,BTCUSDT,DASHUSDT,EOSUSDT,ETCUSDT,ETHUSDT,LINKUSDT,LTCUSDT,NEOUSDT,TRXUSDT,XLMUSDT,XMRUSDT,XRPUSDT,XTZUSDT,ZECUSDT
secid,,,,,,,,,,,,,,,,,,
ADAUSDT,1.2547,0.8810,0.9830,0.8903,1.1589,0.9556,0.9632,0.7157,1.0717,1.0355,0.8707,0.9215,0.7567,0.6908,0.9615,0.7294,1.0997,0.9078
ATOMUSDT,0.8810,1.7238,1.0370,0.9379,1.2230,1.0010,1.0257,0.7455,1.0858,1.0727,0.9130,0.9877,0.8084,0.7224,1.0157,0.7476,1.1380,0.9246
BCHUSDT,0.9830,1.0370,1.7819,1.1358,1.5551,1.2191,1.3392,1.0062,1.4603,1.3050,1.2144,1.1768,0.9221,0.7872,1.1681,0.9405,1.3484,1.1165
BNBUSDT,0.8903,0.9379,1.1358,1.5100,1.3411,1.0293,1.0703,0.7931,1.1930,1.1322,0.9666,1.0200,0.7971,0.7048,1.0551,0.8130,1.1939,0.9529
BTCUSDT,1.1589,1.2230,1.5551,1.3411,2.0221,1.3721,1.4730,1.0708,1.6112,1.5401,1.3357,1.3845,1.0963,0.9182,1.4230,1.0750,1.5803,1.2698
DASHUSDT,0.9556,1.0010,1.2191,1.0293,1.3721,1.9627,1.1618,0.8804,1.2718,1.2056,1.0479,1.0969,0.8382,0.7293,1.1313,0.8736,1.2405,1.1085
EOSUSDT,0.9632,1.0257,1.3392,1.0703,1.4730,1.1618,1.6355,0.9195,1.3805,1.2231,1.1677,1.1403,0.9217,0.7650,1.1439,0.9004,1.2952,1.0615
ETCUSDT,0.7157,0.7455,1.0062,0.7931,1.0708,0.8804,0.9195,1.1320,0.9834,0.9387,0.8507,0.8419,0.6817,0.5747,0.8288,0.6812,0.9533,0.8190
ETHUSDT,1.0717,1.0858,1.4603,1.1930,1.6112,1.2718,1.3805,0.9834,1.7580,1.3568,1.2338,1.2391,0.9784,0.8461,1.2662,0.9993,1.4280,1.1934


In [283]:
ev_rt

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
open_time,,,,,,,,,,,,,,,,,,
2020-04-02 00:00:00,0.0230,-0.0004,0.0013,-0.0003,-0.0026,-0.0025,-0.0018,-0.0011,-0.0009,0.0015,0.0008,0.0011,0.0031,-0.0007,0.0010,0.0022,-0.0033,-0.0023
2020-04-02 00:01:00,0.0193,-0.0007,0.0027,-0.0003,-0.0056,-0.0005,-0.0016,0.0014,0.0006,-0.0023,-0.0001,-0.0012,-0.0018,0.0017,-0.0007,-0.0044,0.0008,0.0007
2020-04-02 00:02:00,0.0044,-0.0003,-0.0017,0.0015,-0.0008,0.0005,0.0010,0.0009,0.0006,-0.0004,0.0004,0.0012,0.0004,0.0016,-0.0004,0.0003,-0.0003,0.0006
2020-04-02 00:03:00,-0.0103,0.0030,0.0003,0.0005,-0.0017,0.0009,0.0017,0.0014,-0.0008,0.0009,-0.0010,-0.0005,0.0002,-0.0014,-0.0013,-0.0003,-0.0012,0.0002
2020-04-02 00:04:00,0.0034,-0.0005,0.0014,-0.0026,0.0010,-0.0003,0.0006,-0.0023,0.0016,-0.0018,0.0010,0.0006,0.0003,0.0006,0.0007,-0.0002,-0.0013,-0.0010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-04-04 23:55:00,-0.0012,0.0001,0.0003,0.0012,-0.0004,0.0005,-0.0001,-0.0004,0.0001,0.0000,-0.0008,0.0005,-0.0004,0.0000,0.0004,-0.0001,-0.0003,-0.0005
2020-04-04 23:56:00,0.0030,0.0002,0.0002,-0.0020,0.0007,-0.0008,-0.0002,-0.0002,0.0006,0.0003,-0.0000,0.0001,0.0004,0.0003,-0.0003,-0.0007,0.0004,0.0005
2020-04-04 23:57:00,0.0004,0.0011,-0.0006,0.0000,-0.0008,0.0002,0.0005,0.0002,-0.0011,-0.0002,0.0010,-0.0004,-0.0011,-0.0002,-0.0002,0.0007,0.0003,-0.0005


In [284]:
ev_rt[1].corr(ev_rt[1].shift(1))

ev_autocorr = ev_rt.apply(lambda x: x.corr(x.shift(1)))
ev_autocorr

tgt_evs = ev_autocorr[ev_autocorr.abs() > .15].index
ev_autocorr[tgt_evs]



-0.19801684951141896

0     0.0632
1    -0.1980
2    -0.2268
3    -0.1759
4    -0.1823
5    -0.1468
6    -0.2882
7    -0.1913
8    -0.2042
9    -0.0507
10   -0.2504
11   -0.1469
12   -0.1595
13   -0.2295
14   -0.1780
15   -0.1130
16   -0.1463
17   -0.1336
dtype: float64

1    -0.1980
2    -0.2268
3    -0.1759
4    -0.1823
6    -0.2882
7    -0.1913
8    -0.2042
10   -0.2504
12   -0.1595
13   -0.2295
14   -0.1780
dtype: float64

In [285]:
ev_reversal_pnls = {}
# for i in range(1, 10):
for i in tgt_evs:
    ev_reversal_pnls[i] = ev_rt[i].apply(np.sign).shift(1).mul(-1).mul(ev_rt[i])
    # ev_reversal_pnls[i] = ev_rt[i].rolling(60* 4).mean().apply(np.sign).shift(2).mul(-1).mul(ev_rt[i])

ev_reversal_pnls = pd.DataFrame(ev_reversal_pnls)

ev_reversal_pnls.apply(perf_stats)
ev_reversal_pnls.cumsum().plot()

perf_stats(ev_reversal_pnls.mean(axis=1))
ev_reversal_pnls.mean(axis=1).cumsum().plot()
# ev_rt[1].apply(np.sign).shift(1).mul(-1).mul(ev_rt[1]).cumsum().plot()

,1,2,3,4,6,7,8,10,12,13,14
sharpe,103.1844,119.8847,96.3702,94.6890,156.9162,125.3794,117.8990,141.9956,76.4010,133.8978,115.9605
sortino,142.6144,172.4954,137.3830,140.2626,259.6884,189.8148,189.4262,226.3191,106.7116,206.3896,177.2633
max_dd,-3.0184,-2.4764,-2.4114,-2.5905,-2.6177,-2.4812,-2.5962,-2.3521,-2.8231,-1.9028,-2.9305
pnl_mean,0.0002,0.0002,0.0001,0.0001,0.0002,0.0001,0.0001,0.0001,0.0001,0.0001,0.0001
pnl_std,0.0011,0.0009,0.0009,0.0009,0.0008,0.0008,0.0008,0.0007,0.0007,0.0006,0.0005
pnl_count,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000
pnl_std_err,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
pnl_tstat,9.3547,10.8687,8.7369,8.5845,14.2260,11.3669,10.6887,12.8733,6.9265,12.1391,10.5129


sharpe         357.9892
sortino        598.7038
max_dd          -2.0501
pnl_mean         0.0001
pnl_std          0.0003
pnl_count     4320.0000
pnl_std_err      0.0000
pnl_tstat       32.4552
dtype: float64

In [286]:
# ev_mod = eigvecs.iloc[:, 1:12].mean(axis=1)
ev_mod = eigvecs.iloc[:, tgt_evs].mean(axis=1)
# ev_mod = eigvecs.mean(axis=1)
ev_mod

secid
ADAUSDT    -0.0693
ATOMUSDT   -0.0802
BCHUSDT    -0.0671
BNBUSDT    -0.0359
BTCUSDT    -0.0055
DASHUSDT    0.0223
EOSUSDT     0.0427
ETCUSDT    -0.0509
ETHUSDT     0.0201
LINKUSDT    0.0606
LTCUSDT     0.0498
NEOUSDT    -0.0431
TRXUSDT    -0.1785
XLMUSDT     0.0315
XMRUSDT    -0.0148
XRPUSDT     0.0353
XTZUSDT     0.0826
ZECUSDT     0.1392
dtype: float64

In [287]:
rt.loc[periods[1].start_time:periods[1].end_time][ev_mod.index].fillna(0)

secid,ADAUSDT,ATOMUSDT,BCHUSDT,BNBUSDT,BTCUSDT,DASHUSDT,EOSUSDT,ETCUSDT,ETHUSDT,LINKUSDT,LTCUSDT,NEOUSDT,TRXUSDT,XLMUSDT,XMRUSDT,XRPUSDT,XTZUSDT,ZECUSDT
open_time,,,,,,,,,,,,,,,,,,
2020-04-05 00:00:00,0.0015,0.0030,0.0026,0.0017,0.0038,0.0018,0.0025,0.0029,0.0032,0.0022,0.0032,0.0028,0.0000,0.0022,0.0026,0.0017,0.0024,0.0015
2020-04-05 00:01:00,-0.0015,-0.0015,-0.0005,-0.0014,-0.0017,-0.0015,-0.0004,-0.0006,-0.0013,-0.0004,-0.0017,0.0000,0.0000,-0.0002,-0.0031,0.0000,0.0000,0.0012
2020-04-05 00:02:00,-0.0006,0.0010,0.0012,0.0018,0.0006,0.0007,0.0013,0.0004,0.0019,0.0000,0.0015,0.0000,0.0017,0.0002,0.0017,0.0006,-0.0006,-0.0009
2020-04-05 00:03:00,0.0006,0.0000,-0.0006,0.0012,-0.0012,-0.0003,0.0000,0.0000,-0.0007,-0.0017,-0.0007,0.0003,-0.0008,-0.0002,-0.0013,-0.0006,0.0000,-0.0003
2020-04-05 00:04:00,0.0000,0.0000,-0.0006,0.0010,-0.0003,-0.0015,-0.0008,-0.0002,-0.0001,-0.0009,0.0002,-0.0004,0.0000,0.0000,0.0000,0.0006,-0.0018,-0.0003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-04-07 23:55:00,-0.0008,0.0004,0.0008,-0.0014,-0.0004,0.0000,-0.0008,-0.0002,-0.0005,0.0011,-0.0004,-0.0003,-0.0008,0.0002,-0.0005,0.0005,-0.0015,-0.0011
2020-04-07 23:56:00,-0.0008,-0.0026,-0.0009,-0.0011,-0.0013,-0.0004,0.0000,-0.0005,-0.0023,-0.0037,-0.0020,-0.0021,0.0000,-0.0012,-0.0012,0.0000,-0.0005,-0.0022
2020-04-07 23:57:00,-0.0006,0.0021,0.0004,0.0002,0.0001,0.0004,0.0000,0.0007,-0.0006,0.0007,0.0016,0.0008,-0.0008,0.0004,0.0000,-0.0021,-0.0005,0.0005


In [288]:
ev_mod

secid
ADAUSDT    -0.0693
ATOMUSDT   -0.0802
BCHUSDT    -0.0671
BNBUSDT    -0.0359
BTCUSDT    -0.0055
DASHUSDT    0.0223
EOSUSDT     0.0427
ETCUSDT    -0.0509
ETHUSDT     0.0201
LINKUSDT    0.0606
LTCUSDT     0.0498
NEOUSDT    -0.0431
TRXUSDT    -0.1785
XLMUSDT     0.0315
XMRUSDT    -0.0148
XRPUSDT     0.0353
XTZUSDT     0.0826
ZECUSDT     0.1392
dtype: float64

In [290]:
ev_mod_rt = rt.loc[periods[1].start_time:periods[1].end_time][ev_mod.index].fillna(0) @ ev_mod
# sig = ev_mod_rt.apply(np.sign).rolling(60*2).mean().mul(-1)
sig = ev_mod_rt.apply(np.sign).rolling(1).mean().mul(-1)
sig.diff().abs().mean()
ev_mod_oos_pnl = sig.shift(1).mul(ev_mod_rt).dropna()

perf_stats(ev_mod_oos_pnl)
ev_mod_oos_pnl.cumsum().plot()

1.1183144246353323

sharpe          68.9898
sortino         92.5472
max_dd          -3.8302
pnl_mean         0.0000
pnl_std          0.0003
pnl_count     4319.0000
pnl_std_err      0.0000
pnl_tstat        6.2539
dtype: float64

In [292]:
ev_mod_oos_pnl.plot.hist(bins=100)

In [273]:
# sig = ev_mod_rt_rm.apply(np.sign).mul(-1).dropna()
sig
turnover = sig.diff()
tcost_model = 0.0015
tcost = turnover.abs().mul(tcost_model)
pnl_gross = sig.shift(1).mul(ev_mod_rt)
pnl_net = pnl_gross.sub(tcost, fill_value=0)

pnl = pd.DataFrame({'pnl_gross': pnl_gross, 'pnl_net': pnl_net, 'turnover': turnover, 'tcost': tcost})
pnl[['pnl_gross', 'pnl_net']].cumsum().plot()

open_time
2020-04-05 00:00:00       NaN
2020-04-05 00:01:00       NaN
2020-04-05 00:02:00       NaN
2020-04-05 00:03:00       NaN
2020-04-05 00:04:00       NaN
                        ...  
2020-04-07 23:55:00   -0.0167
2020-04-07 23:56:00   -0.0000
2020-04-07 23:57:00   -0.0000
2020-04-07 23:58:00   -0.0167
2020-04-07 23:59:00   -0.0000
Length: 4320, dtype: float64

In [105]:
# # regress rt_p['ADAUSDT'] onto ev_rt with statsmodels
# import statsmodels.api as sm

# rt_p = rt_p.dropna(axis=1, thresh=.8)

# rt_p = rt_p.loc[:, ev_mod.index]

# rt_p = rt_p.dropna()
# ev_rt = ev_rt.loc[rt_p.index]
# ev_rt = ev_rt.dropna()

# rt_p = rt_p.loc[ev_rt.index]

# rt_p = sm.add_constant(rt_p)
# model = sm.OLS(endog=rt_p['ADAUSDT'], exog=ev_rt)
# results = model.fit()

# results.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                ADAUSDT   R-squared (uncentered):                   1.000
Model:                            OLS   Adj. R-squared (uncentered):              1.000
Method:                 Least Squares   F-statistic:                          2.114e+31
Date:                Mon, 05 Aug 2024   Prob (F-statistic):                        0.00
Time:                        16:28:15   Log-Likelihood:                      1.6742e+05
No. Observations:                4320   AIC:                                 -3.348e+05
Df Residuals:                    4300   BIC:                                 -3.347e+05
Df Model:                          20                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
0              0.1935   1.16e-17   1.66e+16      0.000       0.193       0.193
1              0.0244   4.51e-17   5.41e+14      0.000       0.024       0.024
2              0.0062   4.77e-17    1.3e+14      0.000       0.006       0.006
3             -0.0281   5.09e-17  -5.53e+14      0.000      -0.028      -0.028
4             -0.0816    5.8e-17  -1.41e+15      0.000      -0.082      -0.082
5             -0.0581   5.88e-17  -9.89e+14      0.000      -0.058      -0.058
6             -0.0036   6.01e-17  -5.94e+13      0.000      -0.004      -0.004
7              0.0259   6.08e-17   4.26e+14      0.000       0.026       0.026
8              0.0904   6.36e-17   1.42e+15      0.000       0.090       0.090
9              0.0525   6.91e-17    7.6e+14      0.000       0.053       0.053
10             0.1503   7.02e-17   2.14e+15      0.000       0.150       0.150
11             0.1969   7.29e-17    2.7e+15      0.000       0.197       0.197
12             0.0239   7.62e-17   3.13e+14      0.000       0.024       0.024
13            -0.8732   8.09e-17  -1.08e+16      0.000      -0.873      -0.873
14            -0.1079   8.19e-17  -1.32e+15      0.000      -0.108      -0.108
15            -0.0720   9.14e-17  -7.87e+14      0.000      -0.072      -0.072
16             0.2555      1e-16   2.54e+15      0.000       0.255       0.255
17            -0.0708   1.03e-16  -6.86e+14      0.000      -0.071      -0.071
18            -0.1622   1.06e-16  -1.53e+15      0.000      -0.162      -0.162
19            -0.0406   1.09e-16  -3.74e+14      0.000      -0.041      -0.041
==============================================================================
Omnibus:                     1462.680   Durbin-Watson:                   1.897
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           100010.211
Skew:                           0.749   Prob(JB):                         0.00
Kurtosis:                      26.524   Cond. No.                         9.33
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [162]:
# calc cumulative residuals for each asset onto each eigenvector

asset_rsid = {}
asset_resid_auto_corr = {}

asset = 'ADAUSDT'
for i in eigvecs.columns:
    # project asset return col onto first i eigenspace
    pred_rt = (ev_rt.iloc[:, :i+1] @ eigvecs.loc[asset].iloc[:i+1])
    resid = rt_p[asset] - pred_rt
    asset_rsid[i] = resid
    asset_resid_auto_corr[i] = resid.corr(resid.shift(1))
    # proj = rt_p[asset] @ first_n_eigvecs
    # proj
    # first_n_eigvecs_mod_rt = (rt_p @ first_n_eigvecs)
    # first_n_eigvects_resid = rt_p[asset] - first_n_eigvecs_mod_rt[asset]
    # break

asset_rsid = pd.DataFrame(asset_rsid)
asset_resid_auto_corr = pd.Series(asset_resid_auto_corr)
asset_rsid.mean().plot()
asset_resid_auto_corr.plot()

In [154]:
resid.corr(resid.shift(1))
rt_p[asset].corr(rt_p[asset].shift(1))

-0.15631180048082963

0.00957652376611724

In [159]:
rt_p[asset].cov(ev_rt[0]) / ev_rt[0].var()

0.19346825964251108

In [160]:
eigvecs

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
secid,,,,,,,,,,,,,,,,,,,,
ADAUSDT,0.1935,0.0244,0.0062,-0.0281,-0.0816,-0.0581,-0.0036,0.0259,0.0904,0.0525,0.1503,0.1969,0.0239,-0.8732,-0.1079,-0.0720,0.2555,-0.0708,-0.1622,-0.0406
ATOMUSDT,0.2066,0.0719,0.0305,-0.1482,-0.4010,-0.5726,-0.3723,0.4385,-0.1272,-0.2155,-0.1164,-0.0952,-0.0903,0.0912,0.0202,-0.0265,0.0416,-0.0527,-0.0001,-0.0491
BCHUSDT,0.2467,-0.1030,0.1525,0.0779,0.2159,0.1902,-0.1670,-0.0363,-0.1433,-0.2775,-0.1245,-0.1648,0.0373,0.0414,0.0059,-0.6362,0.2762,0.3842,-0.0866,0.0876
BNBUSDT,0.2126,-0.0380,0.0501,0.0306,0.0701,-0.0713,-0.0667,-0.0733,-0.0659,-0.1034,-0.1167,0.8431,0.2728,0.2268,-0.1059,0.1220,0.0733,0.1371,-0.0092,-0.1201
BTCUSDT,0.2814,-0.0770,0.0874,0.0546,0.1335,-0.0369,-0.0586,-0.0810,-0.0443,-0.0254,-0.0423,0.0626,0.1035,0.0411,0.1060,-0.0959,-0.2238,-0.5711,-0.2394,0.6327
DASHUSDT,0.2302,-0.0900,0.2085,0.1768,-0.4701,0.3107,0.4665,0.3096,-0.4358,0.1497,0.0995,0.0097,0.0837,0.0510,0.0217,-0.0066,-0.0158,0.0177,0.0025,0.0136
EOSUSDT,0.2365,-0.0559,0.1274,0.0451,0.1278,0.1139,-0.1951,-0.0688,-0.1391,-0.1303,-0.0226,-0.2110,0.0749,-0.0985,0.3333,0.6878,0.2815,0.2185,0.0766,0.1919
ETCUSDT,0.1753,-0.0573,0.1097,0.0702,0.1007,0.1268,-0.0977,0.0308,-0.0159,-0.1169,0.0363,-0.1739,-0.1039,0.0650,-0.8799,0.2206,0.0372,-0.1239,0.1079,0.0245
ETHUSDT,0.2573,-0.0631,0.1088,0.0423,0.1091,0.1157,-0.0871,-0.0902,-0.0469,-0.1233,-0.0370,-0.0095,-0.0031,-0.1092,0.2088,-0.1323,-0.1167,-0.4077,0.6726,-0.3887


In [143]:
for i in range(1, 20):
    (ev_rt.iloc[:, :i] @ eigvecs.loc['ADAUSDT'].iloc[:i]).corr(rt_p['ADAUSDT'])


0.8082262524534957

0.8086550963517672

0.8086798194889052

0.8091263661508604

0.8120171030189107

...

0.9880666940411253

0.988808251544715

0.9965125264199486

0.9970713277422087

0.9998346937713185

In [114]:
rt_p

,const,ADAUSDT,ATOMUSDT,BCHUSDT,BNBUSDT,BTCUSDT,DASHUSDT,EOSUSDT,ETCUSDT,ETHUSDT,...,LINKUSDT,LTCUSDT,NEOUSDT,TRXUSDT,VETUSDT,XLMUSDT,XMRUSDT,XRPUSDT,XTZUSDT,ZECUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2020-04-02 00:00:00,1.0000,0.0049,0.0056,0.0053,0.0054,0.0066,0.0062,0.0058,0.0022,0.0114,...,0.0057,0.0056,0.0057,0.0052,0.0099,0.0039,0.0017,0.0057,0.0062,0.0032
2020-04-02 00:01:00,1.0000,0.0019,0.0040,0.0092,0.0043,0.0039,0.0032,0.0093,0.0070,0.0034,...,0.0031,0.0058,0.0045,0.0043,0.0047,0.0042,0.0019,0.0034,0.0056,0.0035
2020-04-02 00:02:00,1.0000,0.0019,0.0020,0.0000,0.0003,0.0014,0.0008,0.0026,0.0008,0.0012,...,-0.0004,0.0005,0.0007,0.0000,0.0016,0.0010,0.0014,0.0006,0.0012,0.0029
2020-04-02 00:03:00,1.0000,-0.0019,-0.0030,-0.0015,-0.0013,-0.0035,-0.0044,-0.0031,-0.0024,-0.0020,...,-0.0035,-0.0040,-0.0041,-0.0009,-0.0066,-0.0019,-0.0031,-0.0023,0.0000,-0.0009
2020-04-02 00:04:00,1.0000,0.0000,0.0000,0.0004,-0.0009,0.0003,-0.0005,0.0013,0.0008,0.0021,...,0.0035,0.0013,0.0038,0.0000,0.0022,0.0007,0.0008,0.0006,0.0000,-0.0003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-04-04 23:55:00,1.0000,0.0000,-0.0010,-0.0004,-0.0001,-0.0008,0.0003,-0.0004,0.0000,0.0001,...,-0.0013,0.0000,0.0000,0.0000,-0.0006,0.0000,-0.0004,-0.0011,0.0000,0.0000
2020-04-04 23:56:00,1.0000,0.0006,0.0010,0.0010,0.0012,0.0011,0.0000,0.0013,0.0000,0.0000,...,0.0026,0.0005,0.0011,0.0000,0.0006,0.0005,0.0006,0.0006,0.0006,-0.0003
2020-04-04 23:57:00,1.0000,0.0003,0.0005,-0.0003,-0.0004,-0.0002,-0.0007,-0.0008,0.0012,0.0000,...,-0.0004,0.0002,-0.0006,0.0008,0.0000,-0.0002,0.0002,0.0011,0.0012,0.0003


In [333]:
# ev_sprd = ev_rt[1] - ev_rt[3]
ev_sprd = ev_rt[8]

ev_sprd_pnls = {}
for i in range(1, 10):
    bwd = 60 * 1
    fwd = 20

    ev_sprd = ev_rt[i]

    ev_sprd_rm_60= ev_sprd.rolling(bwd*2).mean()
    ev_sprd_rm_5 = ev_sprd.rolling(fwd).mean()

    ev_sprd_rm_60.corr(ev_sprd_rm_5.shift(-fwd))

    # ev_sprd.apply(np.sign).shift(1).mul(-1).mul(ev_sprd).cumsum().plot()
    sprd_sig = ev_sprd_rm_60.apply(np.sign).shift(1).mul(-1).rolling(fwd).mean()
    # sprd_sig.mul(ev_sprd).cumsum().plot()
    sprd_sig.diff().abs().mean()

    turnover = sprd_sig.diff()
    tcost_model = 0.0015
    tcost = turnover.abs().mul(tcost_model)
    pnl_gross = sprd_sig.shift(1).mul(ev_sprd)
    pnl_net = pnl_gross.sub(tcost, fill_value=0)

    pnl = pd.DataFrame({'pnl_gross': pnl_gross, 'pnl_net': pnl_net, 'turnover': turnover, 'tcost': tcost})

    ev_sprd_pnls[i] = pnl_gross

pnl[['pnl_gross', 'pnl_net']].apply(perf_stats)
pnl[['pnl_gross', 'pnl_net']].cumsum().plot()

-0.050122790667665715

0.02485645933014354

-0.20292267980840012

0.02799043062200957

-0.20466291730320227

...

-0.21633438754168266

0.024354066985645934

-0.1625317912590885

0.024545454545454544

,pnl_gross,pnl_net
sharpe,13.9742,-27.1899
sortino,18.8723,-38.7263
max_dd,-3.1389,-3.1389
pnl_mean,0.0000,-0.0000
pnl_std,0.0006,0.0007
pnl_count,4320.0000,4320.0000
pnl_std_err,0.0000,0.0000
pnl_tstat,1.2669,-2.4650


In [336]:
ev_sprd_pnls = pd.DataFrame(ev_sprd_pnls)
ev_sprd_pnls.apply(perf_stats)

ev_sprd_pnls.mean(axis=1).cumsum().plot()

,1,2,3,4,5,6,7,8,9
sharpe,10.6910,21.1807,21.1354,-1.2798,6.4876,4.4287,8.1181,13.2553,13.9742
sortino,14.7079,29.7984,28.5935,-1.7543,8.2563,6.1657,11.4530,18.7246,18.8723
max_dd,-1.8332,-56.7682,-2.4295,-2.6700,-5.8506,-2.5833,-2.4486,-3.8247,-3.1389
pnl_mean,0.0000,0.0000,0.0000,-0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
pnl_std,0.0010,0.0008,0.0008,0.0008,0.0008,0.0008,0.0007,0.0007,0.0006
pnl_count,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000,4320.0000
pnl_std_err,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
pnl_tstat,0.9692,1.9202,1.9161,-0.1160,0.5882,0.4015,0.7360,1.2017,1.2669


In [103]:
sm.OLS?

Init signature: sm.OLS(endog, exog=None, missing='none', hasconst=None, **kwargs)
Docstring:     
Ordinary Least Squares

Parameters
----------
endog : array_like
    A 1-d endogenous response variable. The dependent variable.
exog : array_like
    A nobs x k array where `nobs` is the number of observations and `k`
    is the number of regressors. An intercept is not included by default
    and should be added by the user. See
    :func:`statsmodels.tools.add_constant`.
missing : str
    Available options are 'none', 'drop', and 'raise'. If 'none', no nan
    checking is done. If 'drop', any observations with nans are dropped.
    If 'raise', an error is raised. Default is 'none'.
hasconst : None or bool
    Indicates whether the RHS includes a user-supplied constant. If True,
    a constant is not checked for and k_constant is set to 1 and all
    result statistics are calculated as if a constant is present. If
    False, a constant is not checked for and k_constant is set to 0.
**kwa